<div align="center">

# Trivia Night with KONASH

**Train a 4B model to answer multi-constraint trivia by searching Wikipedia**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/konaequity/konash/blob/main/notebooks/trivia_night.ipynb)

</div>

---

Run all cells via **Runtime → Run all**. Requires a Colab GPU runtime — go to **Runtime → Change runtime type → T4 GPU** before running.

This notebook trains a **Qwen 3 4B** model to answer multi-constraint trivia questions. Inference (QA synthesis and rollouts) runs on **Groq's free API** (Qwen3-32B) for speed, while OAPL gradient updates run on the local T4 GPU. Training takes ~10 minutes.

**Setup:** Add your Groq API key as a Colab secret named `GROQ_API_KEY` (key icon in the left sidebar). Get a free key at [console.groq.com](https://console.groq.com).

---

In [ ]:
!pip install -qU "konash[train] @ git+https://github.com/konaequity/konash.git"

In [ ]:
import os
import json
import urllib.request
import urllib.parse

import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. Training will be very slow.")
    print("Go to Runtime > Change runtime type > Select GPU.")

In [ ]:
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    print("Groq API key loaded from Colab secrets.")
except (ImportError, Exception):
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
    if GROQ_API_KEY:
        print("Groq API key loaded from environment.")
    else:
        raise RuntimeError(
            "No GROQ_API_KEY found. Add it as a Colab secret "
            "(key icon in sidebar) or set it as an environment variable."
        )

## Agentic Environment

The environment defines the knowledge the agent can search over. We fetch 8 Wikipedia articles covering the topics our trivia questions target and write them to disk as the agent's searchable corpus.

In [ ]:
def fetch_wikipedia_article(title: str) -> str:
    """Fetch plain-text extract of a Wikipedia article via the API."""
    params = urllib.parse.urlencode({
        "action": "query",
        "titles": title,
        "prop": "extracts",
        "explaintext": "1",
        "exsectionformat": "plain",
        "format": "json",
    })
    url = f"https://en.wikipedia.org/w/api.php?{params}"
    req = urllib.request.Request(url, headers={"User-Agent": "KONASH-TriviaNotebook/1.0"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        data = json.loads(resp.read())
    pages = data.get("query", {}).get("pages", {})
    for page in pages.values():
        return page.get("extract", "")
    return ""


TRIVIA_TOPICS = [
    "Ancient Rome",
    "Rosetta Stone",
    "Periodic table",
    "DNA",
    "Mount Everest",
    "Ring of Fire",
    "Leonardo da Vinci",
    "Chess",
]

CORPUS_DIR = "./trivia_corpus"
os.makedirs(CORPUS_DIR, exist_ok=True)

print(f"Fetching {len(TRIVIA_TOPICS)} Wikipedia articles...")
for i, topic in enumerate(TRIVIA_TOPICS):
    text = fetch_wikipedia_article(topic)
    if text and len(text) > 200:
        words = text.split()
        if len(words) > 3000:
            text = " ".join(words[:3000])
        fname = topic.replace(" ", "_").replace("/", "_") + ".txt"
        with open(os.path.join(CORPUS_DIR, fname), "w") as f:
            f.write(text)
        print(f"  [{i+1}/{len(TRIVIA_TOPICS)}] {topic}: {len(text):,} chars")
    else:
        print(f"  [{i+1}/{len(TRIVIA_TOPICS)}] {topic}: SKIPPED (too short)")

## Create the Agent

We create a `konash.Agent` in **split mode**: inference (synthesis, rollouts, eval) runs on Groq's Qwen3-32B for speed, while the local Qwen3-4B with 4-bit QLoRA handles OAPL gradient updates on the T4.

In [ ]:
import konash

agent = konash.Agent(
    base_model="Qwen/Qwen3-4B",
    corpus=CORPUS_DIR,
    project="trivia-night",
    load_in_4bit=True,
    gradient_checkpointing=True,
    # Groq for fast inference (synthesis + rollouts + solve)
    inference_api_base="https://api.groq.com/openai/v1",
    inference_api_key=GROQ_API_KEY,
    inference_model="qwen/qwen3-32b",
)

## Train

The agent synthesizes multi-constraint trivia questions via Groq (fast), generates search-and-answer rollouts, filters to the learning frontier by pass rate, and trains with OAPL on the local T4. One iteration takes ~10 minutes on Groq's free tier.

We provide **few-shot examples** covering the six KARLBench capability types to guide the synthesizer toward diverse, difficult questions (KARL Section 4.1). These examples show the format and difficulty level — the synthesizer will generate new questions grounded in the corpus.

In [ ]:
from konash.synthesis.qa import SyntheticExample

# One example per KARLBench capability type (KARL Table 1, Section 4.1).
# All answers verified against the Wikipedia corpus articles.
FEW_SHOT_EXAMPLES = [
    # Type 1: Constraint-driven entity search
    # Multiple biographical constraints narrowing to a single entity
    SyntheticExample(
        question="Which Renaissance polymath, born in Vinci in 1452, was apprenticed to Andrea del Verrocchio in Florence and later served as 'premier painter and engineer and architect of the King' under Francis I of France?",
        answer="Leonardo da Vinci was born in Vinci in 1452, apprenticed to Verrocchio in Florence, and served Francis I as premier painter and engineer and architect of the King.",
    ),
    # Type 2: Cross-document report synthesis
    # Requires combining facts from Rosetta Stone + Ancient Rome articles
    SyntheticExample(
        question="The Rosetta Stone was created during the Ptolemaic dynasty, which ended when a Roman leader annexed Egypt. Who was that Roman leader, and which French officer discovered the Rosetta Stone centuries later during Napoleon's campaign?",
        answer="The Ptolemaic dynasty ended when Octavian (later Augustus) annexed Egypt after the Battle of Actium in 30 BC. The Rosetta Stone was discovered in 1799 by French officer Pierre-François Bouchard during Napoleon's Egyptian campaign.",
    ),
    # Type 3: Long-document numerical reasoning
    # Requires extracting and comparing specific numbers from Mount Everest article
    SyntheticExample(
        question="The first Survey of India measurement of Mount Everest's height was 8,840 m. By how many metres did the 2020 Chinese/Nepalese survey revise this figure upward, and how heavy were the theodolites carried by the original survey teams?",
        answer="The 2020 survey established Everest's height as 8,848.86 m, which is 8.86 m higher than the original 8,840 m measurement. The original survey teams carried theodolites weighing 500 kg (1,100 lb) each, requiring 12 men to carry.",
    ),
    # Type 4: Exhaustive entity retrieval
    # Must find ALL qualifying eruptions, not just one
    SyntheticExample(
        question="Name four of the largest Holocene-era volcanic eruptions that occurred along the Pacific Ring of Fire, including the specific volcanoes and their locations.",
        answer="Four of the largest Holocene eruptions along the Ring of Fire include: Mount Tambora (Indonesia, 1815), Krakatoa (Indonesia, 1883), Mount Pinatubo (Philippines, 1991), and Novarupta (Alaska, 1912).",
    ),
    # Type 5: Procedural reasoning over sequential events
    # Requires tracing a chain of custody through time
    SyntheticExample(
        question="Trace the chain of custody of the Rosetta Stone from its discovery to its current location: who found it, who surrendered it under which treaty, and where is it housed today?",
        answer="Pierre-François Bouchard discovered the stone in 1799 during Napoleon's campaign. After the French defeat, it was surrendered to the British under the Treaty of Alexandria (also called the Capitulation of Alexandria) in 1801. It has been on continuous public display at the British Museum since 1802.",
    ),
    # Type 6: Fact aggregation across scattered mentions
    # Requires gathering multiple names from different sections of the article
    SyntheticExample(
        question="Mount Everest has been known by several names across different cultures and surveys. List at least three distinct names or designations that have been used for the mountain, and their origins.",
        answer="The mountain has been known as: Peak XV (its original Survey of India designation), Chomolungma (Tibetan name meaning 'Goddess Mother of the World'), Sagarmāthā (Nepali name meaning 'Peak of Heaven'), and Mount Everest (named after George Everest, the British Surveyor General of India).",
    ),
]

In [ ]:
results = agent.train(
    iterations=1,
    rollouts_per_example=4,
    max_examples=5,
    few_shot_examples=FEW_SHOT_EXAMPLES,
)

## Play Trivia

The trained agent can now answer trivia questions by searching the Wikipedia corpus. Let's try it on a few questions it has never seen.

In [ ]:
TRIVIA_QUESTIONS = [
    # Frontier models get these wrong — answers require deep retrieval
    "What organization produced the first English translation of the Rosetta Stone's Greek text, and in what year?",
    "Who was the first person to publish the electron orbital filling rule known as the Aufbau principle, and in what year was it published?",
    "In Ancient Rome, wood planks from which specific mountain range were discovered 1,700 km away in Central Rome, and in what date range were the trees felled?",
    # More retrieval-hard questions (uncomment to try):
    # "Who was the first person to identify Mount Everest as the world's highest peak, and from which city was he stationed when he made the calculation?",
    # "What is the estimated number of legal positions in chess, expressed with its specific coefficient and confidence interval?",
    # "In the 1993 eruption of Lascar volcano along the Ring of Fire, how far did pyroclastic flows travel from the summit, and how far away was ash fall recorded?",
    # "What is the exact length in centimeters of the total female diploid nuclear genome per human cell?",
    # "What specific mathematical proof did Leonardo da Vinci contribute to, involving the doubling of a square's area, and in which of his notebooks does it appear?",
]

for q in TRIVIA_QUESTIONS:
    answer = agent.solve(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

---

Your trivia agent is trained and ready! Check out the [KONASH GitHub](https://github.com/konaequity/konash) for more examples.

**Next steps:**
- Add more Wikipedia articles to the corpus for broader coverage
- Run a 3rd training iteration for further improvement
- Deploy with vLLM + LoRA hot-swapping for production serving